In [2]:
# Load the Data
import pandas as pd

data = {}

path = "Parcels-CSV-2017-2025"

for year in range(2017, 2026):
    file = f"{path}/PAR{year}.csv"

    data[year] = pd.read_csv(
        file,
        encoding="utf-8-sig",
        encoding_errors="replace",
        low_memory=False
    )

    print(year, data[year].shape)

2017 (127917, 80)
2018 (127782, 80)
2019 (127611, 80)
2020 (127647, 80)
2021 (127355, 76)
2022 (127327, 76)
2023 (127105, 76)
2024 (127084, 46)
2025 (127025, 46)


In [4]:
# Check Data Quality
for year in range(2017, 2026):
    df = data[year]

    df["ASMTTOTAL"] = pd.to_numeric(
        df["ASMTTOTAL"], errors="coerce"
    )

    duplicates = df["HANDLE"].duplicated().sum()
    zero = (df["ASMTTOTAL"] == 0).sum()
    negative = (df["ASMTTOTAL"] < 0).sum()
    missing = df["ASMTTOTAL"].isnull().sum()

    print(year, duplicates, zero, negative, missing)

2017 0 710 0 0
2018 0 717 0 0
2019 0 692 1 0
2020 0 791 1 0
2021 0 1226 0 0
2022 0 6381 0 0
2023 0 6554 0 1
2024 0 6366 0 0
2025 0 6224 0 0


In [6]:
# Correct the Property Matching
values = pd.concat(
    {
        year: data[year].set_index("HANDLE")["ASMTTOTAL"]
        for year in range(2017, 2026)
    },
    axis=1
)

display(values.head())

zero_2022 = values[values[2022] == 0]

print("Zero in 2022:", len(zero_2022))
print("Positive in 2021:", (zero_2022[2021] > 0).sum())
print("Zero in both years:",
      ((zero_2022[2021] == 0) &
       (zero_2022[2022] == 0)).sum())
print("Not found in 2021:", zero_2022[2021].isna().sum())

,2017,2018,2019,2020,2021,2022,2023,2024,2025
HANDLE,,,,,,,,,
1.000100e+10,4400.0,4400.0,4400.0,4500.0,6800.0,0.0,0.0,0.0,0.0
1.000100e+10,19000.0,19000.0,19000.0,19000.0,19000.0,0.0,0.0,0.0,0.0
1.000100e+10,6500.0,6500.0,6500.0,6500.0,6500.0,0.0,0.0,0.0,0.0
1.000100e+10,1200.0,1200.0,1200.0,1200.0,1200.0,1200.0,1200.0,1200.0,1200.0
1.000100e+10,4000.0,4000.0,4000.0,4000.0,4000.0,4000.0,4000.0,4000.0,4000.0


Zero in 2022: 6381
Positive in 2021: 5205
Zero in both years: 1150
Not found in 2021: 26


In [16]:
# Create the Residential Modeling Dataset

model_data = []
selection_summary = []

for year in range(2017, 2025):
    df = data[year].copy()

    df["Year"] = year

    # Identify residential properties
    land_use = pd.to_numeric(df["LANDUSE1"], errors="coerce")

    built_residential = (
        land_use.between(1110, 1199)
        & ~land_use.isin([1116, 1145, 1155])
    )

    # Match next-year assessed values
    df["Next_ASMTTOTAL"] = df["HANDLE"].map(values[year + 1])

    # Require positive assessed values in both years
    valid_values = (
        (df["ASMTTOTAL"] > 0) &
        (df["Next_ASMTTOTAL"] > 0)
    )

    # Record sample selection
    selection_summary.append({
        "Year": year,
        "Total Parcels": len(df),
        "Built Residential": built_residential.sum(),
        "Valid Residential Pairs": (
            built_residential & valid_values
        ).sum()
    })

    # Keep eligible residential observations
    df = df[built_residential & valid_values].copy()

    df["Appreciation"] = (
        df["Next_ASMTTOTAL"] - df["ASMTTOTAL"]
    ) / df["ASMTTOTAL"]

    model_data.append(df)

model_data = pd.concat(model_data, ignore_index=True)

selection_summary = pd.DataFrame(selection_summary)

display(selection_summary)

print("Total Residential Observations:", len(model_data))

,Year,Total Parcels,Built Residential,Valid Residential Pairs
0,2017,127917,95956,95786
1,2018,127782,95801,95666
2,2019,127611,95739,95628
3,2020,127647,95594,95015
4,2021,127355,94968,93675
5,2022,127327,95026,93498
6,2023,127105,94488,93282
7,2024,127084,94431,93230


Total Residential Observations: 755780


In [17]:
# Investigate Residential Sample Exclusions

exclusion_summary = []

for year in range(2017, 2025):
    df = data[year].copy()

    land_use = pd.to_numeric(
        df["LANDUSE1"], errors="coerce"
    )

    built = (
        land_use.between(1110, 1199)
        & ~land_use.isin([1116, 1145, 1155])
    )

    # Step A: Current assessed value must be positive
    current_valid = built & (df["ASMTTOTAL"] > 0)

    # Step B: The parcel must appear in the next year
    next_exists = df["HANDLE"].isin(
        data[year + 1]["HANDLE"]
    )
    matched = current_valid & next_exists

    # Step C: Next-year assessed value must be positive
    next_value = df["HANDLE"].map(values[year + 1])
    final_valid = matched & (next_value > 0)

    exclusion_summary.append({
        "Year": year,
        "Built Residential": built.sum(),
        "Invalid Current Value": (built & ~current_valid).sum(),
        "No Next-Year Match": (current_valid & ~next_exists).sum(),
        "Invalid Next Value": (matched & ~(next_value > 0)).sum(),
        "Final Valid Pairs": final_valid.sum()
    })

exclusion_summary = pd.DataFrame(exclusion_summary)

display(exclusion_summary)

,Year,Built Residential,Invalid Current Value,No Next-Year Match,Invalid Next Value,Final Valid Pairs
0,2017,95956,3,167,0,95786
1,2018,95801,5,129,1,95666
2,2019,95739,4,85,22,95628
3,2020,95594,30,254,295,95015
4,2021,94968,308,95,890,93675
5,2022,95026,1145,130,253,93498
6,2023,94488,1113,80,13,93282
7,2024,94431,1037,92,72,93230


In [22]:
# Finalize the Primary Residential Sample

# Preserve the original built-residential dataset
model_data_built = model_data.copy()

# Standardize vacant-land status
vacant_status = (
    model_data_built["VACANTLAND"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Keep non-vacant residential properties
primary_eligible = (vacant_status == "N").fillna(False)

model_data = model_data_built.loc[
    primary_eligible
].copy()

print("Original Built Residential:", len(model_data_built))
print("Primary Residential Sample:", len(model_data))
print("Excluded from Primary Sample:",
      len(model_data_built) - len(model_data))

display(model_data.groupby("Year").size())

Original Built Residential: 755780
Primary Residential Sample: 730611
Excluded from Primary Sample: 25169


Year
2017    92669
2018    92235
2019    91724
2020    91605
2021    90747
2022    90805
2023    90465
2024    90361
dtype: int64

In [23]:
# Assessment Change Distribution by Year
import numpy as np

assessment_data = model_data.copy()

# Calculate one-year log appreciation
assessment_data["Log_Appreciation"] = (
    np.log(assessment_data["Next_ASMTTOTAL"])
    - np.log(assessment_data["ASMTTOTAL"])
)

# Identify properties with no assessed value change
assessment_data["Unchanged"] = (
    assessment_data["ASMTTOTAL"]
    == assessment_data["Next_ASMTTOTAL"]
)

# Identify extreme appreciation changes
assessment_data["Extreme"] = (
    (assessment_data["Appreciation"] > 1.0) |
    (assessment_data["Appreciation"] < -0.5)
)

# Summarize assessment changes by year
assessment_summary = assessment_data.groupby("Year").agg(
    Total_Records=("HANDLE", "size"),
    Unchanged_Count=("Unchanged", "sum"),
    Extreme_Count=("Extreme", "sum"),
    Mean_Log_Appreciation=("Log_Appreciation", "mean"),
    Median_Log_Appreciation=("Log_Appreciation", "median")
)

assessment_summary["Unchanged_%"] = (
    100 * assessment_summary["Unchanged_Count"]
    / assessment_summary["Total_Records"]
)

assessment_summary["Extreme_%"] = (
    100 * assessment_summary["Extreme_Count"]
    / assessment_summary["Total_Records"]
)

display(assessment_summary.round(3))

,Total_Records,Unchanged_Count,Extreme_Count,Mean_Log_Appreciation,Median_Log_Appreciation,Unchanged_%,Extreme_%
Year,,,,,,,
2017,92669,91043,551,0.008,0.000,98.245,0.595
2018,92235,3962,1074,0.099,0.076,4.296,1.164
2019,91724,85006,495,0.001,0.000,92.676,0.540
2020,91605,18391,1014,0.049,0.051,20.076,1.107
2021,90747,89127,448,0.004,0.000,98.215,0.494
2022,90805,3592,1089,0.078,0.080,3.956,1.199
2023,90465,89070,375,0.007,0.000,98.458,0.415
2024,90361,3172,5559,0.175,0.116,3.510,6.152


In [24]:
# Quantify the Two-Year Assessment Cycle

assessment_data["Target_Year"] = (
    assessment_data["Year"] + 1
)

assessment_data["Assessment_Cycle"] = np.where(
    assessment_data["Target_Year"] % 2 == 1,
    "Odd Year (Reassessment)",
    "Even Year"
)

# Use development-period observations only
cycle_data = assessment_data[
    assessment_data["Year"] <= 2023
].copy()

cycle_summary = cycle_data.groupby(
    "Assessment_Cycle"
).agg(
    Total_Records=("HANDLE", "size"),
    Unchanged_Count=("Unchanged", "sum"),
    Extreme_Count=("Extreme", "sum"),
    Mean_Log_Appreciation=("Log_Appreciation", "mean"),
    Median_Log_Appreciation=("Log_Appreciation", "median")
)

cycle_summary["Unchanged_%"] = (
    100 * cycle_summary["Unchanged_Count"]
    / cycle_summary["Total_Records"]
)

cycle_summary["Extreme_%"] = (
    100 * cycle_summary["Extreme_Count"]
    / cycle_summary["Total_Records"]
)

display(cycle_summary.round(3))

,Total_Records,Unchanged_Count,Extreme_Count,Mean_Log_Appreciation,Median_Log_Appreciation,Unchanged_%,Extreme_%
Assessment_Cycle,,,,,,,
Even Year,365605,354246,1869,0.005,0.000,96.893,0.511
Odd Year (Reassessment),274645,25945,3177,0.075,0.065,9.447,1.157


In [25]:
# Investigate Zero Assessment Transitions

zero_transition_summary = []

for year in range(2019, 2024):

    df = data[year].copy()

    land_use = pd.to_numeric(
        df["LANDUSE1"], errors="coerce"
    )

    vacant_status = (
        df["VACANTLAND"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    # Use the proposed primary residential eligibility rule
    residential = (
        land_use.between(1110, 1199)
        & ~land_use.isin([1116, 1145, 1155])
        & (vacant_status == "N")
    ).fillna(False)

    current_value = pd.to_numeric(
        df["ASMTTOTAL"], errors="coerce"
    )

    next_value = pd.to_numeric(
        df["HANDLE"].map(values[year + 1]),
        errors="coerce"
    )

    # Start with positive-value residential properties
    eligible = residential & (current_value > 0)

    matched = eligible & next_value.notna()

    positive_to_zero = matched & (next_value == 0)

    zero_transition_summary.append({
        "Transition": f"{year} to {year + 1}",
        "Eligible Current Parcels": int(eligible.sum()),
        "Matched with Next Value": int(matched.sum()),
        "Positive to Zero": int(positive_to_zero.sum()),
        "Positive to Zero %": (
            100 * positive_to_zero.sum() / matched.sum()
            if matched.sum() > 0 else np.nan
        )
    })

zero_transition_summary = pd.DataFrame(
    zero_transition_summary
)

display(zero_transition_summary.round(3))

,Transition,Eligible Current Parcels,Matched with Next Value,Positive to Zero,Positive to Zero %
0,2019 to 2020,91814,91745,21,0.023
1,2020 to 2021,92014,91891,286,0.311
2,2021 to 2022,91362,91281,534,0.585
3,2022 to 2023,91161,91039,234,0.257
4,2023 to 2024,90550,90478,13,0.014


In [31]:
# Step 3.1: Within-Property Lag-1 Correlation
dependence_data = assessment_data[
    assessment_data["Year"] <= 2023
].copy().sort_values(["HANDLE", "Year"])

dependence_data["Previous_Log_Appreciation"] = (
    dependence_data.groupby("HANDLE")["Log_Appreciation"].shift(1)
)
dependence_data["Previous_Year"] = (
    dependence_data.groupby("HANDLE")["Year"].shift(1)
)
consecutive = dependence_data.loc[
    dependence_data["Year"] == dependence_data["Previous_Year"] + 1
].copy()
lag1_corr = consecutive[
    ["Log_Appreciation", "Previous_Log_Appreciation"]
].corr().iloc[0, 1]
print("Consecutive Property-Year Pairs:", len(consecutive))
print("Unique Properties:", consecutive["HANDLE"].nunique())
print("Lag-1 Correlation:", round(lag1_corr, 4))

Consecutive Property-Year Pairs: 545594
Unique Properties: 93722
Lag-1 Correlation: -0.0807


In [32]:
# Step 3.2: Lag-1 Correlation After Removing Year Effects
dependence_data["Year_Mean"] = (
    dependence_data.groupby("Year")["Log_Appreciation"].transform("mean")
)
dependence_data["Year_Centered_Log"] = (
    dependence_data["Log_Appreciation"] - dependence_data["Year_Mean"]
)
dependence_data["Previous_Year_Centered_Log"] = (
    dependence_data.groupby("HANDLE")["Year_Centered_Log"].shift(1)
)
consecutive_adjusted = dependence_data.loc[
    dependence_data["Year"] == dependence_data["Previous_Year"] + 1
].copy()
adjusted_corr = consecutive_adjusted[
    ["Year_Centered_Log", "Previous_Year_Centered_Log"]
].corr().iloc[0, 1]
print("Original Lag-1 Correlation:", round(lag1_corr, 4))
print("Year-Adjusted Lag-1 Correlation:", round(adjusted_corr, 4))
print("Consecutive Pairs:", len(consecutive_adjusted))

Original Lag-1 Correlation: -0.0807
Year-Adjusted Lag-1 Correlation: -0.034
Consecutive Pairs: 545594


In [33]:
# Step 3.3: Lag-1 Correlation Adjusted for Year and Property Type
land_use = pd.to_numeric(dependence_data["LANDUSE1"], errors="coerce")
major_codes = [1110, 1120, 1130, 1140, 1185]
dependence_data["Property_Type"] = (
    land_use.where(land_use.isin(major_codes), -1).astype("int64")
)
dependence_data["Year_Class_Mean"] = (
    dependence_data.groupby(["Year", "Property_Type"])["Log_Appreciation"]
    .transform("mean")
)
dependence_data["Year_Class_Centered_Log"] = (
    dependence_data["Log_Appreciation"] - dependence_data["Year_Class_Mean"]
)
dependence_data["Previous_Year_Class_Log"] = (
    dependence_data.groupby("HANDLE")["Year_Class_Centered_Log"].shift(1)
)
consecutive_class = dependence_data.loc[
    dependence_data["Year"] == dependence_data["Previous_Year"] + 1
].copy()
class_adjusted_corr = consecutive_class[
    ["Year_Class_Centered_Log", "Previous_Year_Class_Log"]
].corr().iloc[0, 1]
print("Original Correlation:", round(lag1_corr, 4))
print("Year-Adjusted Correlation:", round(adjusted_corr, 4))
print("Year + Class Adjusted Correlation:", round(class_adjusted_corr, 4))

Original Correlation: -0.0807
Year-Adjusted Correlation: -0.034
Year + Class Adjusted Correlation: -0.0337


In [34]:
# Step 3.4: Within-Property Lag-2 Correlation
lag2_data = dependence_data.sort_values(["HANDLE", "Year"]).copy()
lag2_data["Previous_Log_2"] = (
    lag2_data.groupby("HANDLE")["Log_Appreciation"].shift(2)
)
lag2_data["Previous_Year_2"] = (
    lag2_data.groupby("HANDLE")["Year"].shift(2)
)
lag2_data["Previous_Adjusted_2"] = (
    lag2_data.groupby("HANDLE")["Year_Class_Centered_Log"].shift(2)
)
lag2_pairs = lag2_data.loc[
    lag2_data["Year"] == lag2_data["Previous_Year_2"] + 2
].copy()
lag2_raw = lag2_pairs[
    ["Log_Appreciation", "Previous_Log_2"]
].corr().iloc[0, 1]
lag2_adjusted = lag2_pairs[
    ["Year_Class_Centered_Log", "Previous_Adjusted_2"]
].corr().iloc[0, 1]
print("Lag-2 Property-Year Pairs:", len(lag2_pairs))
print("Unique Properties:", lag2_pairs["HANDLE"].nunique())
print("Raw Lag-2 Correlation:", round(lag2_raw, 4))
print("Year + Class Adjusted Lag-2:", round(lag2_adjusted, 4))

Lag-2 Property-Year Pairs: 451748
Unique Properties: 92651
Raw Lag-2 Correlation: 0.0546
Year + Class Adjusted Lag-2: 0.0184


In [37]:
# Step 3.7: Geographic Differences by WARD (2022 to 2023)
ward_data = assessment_data.loc[
    assessment_data["Year"] == 2022,
    ["WARD", "Log_Appreciation"]
].copy()
ward_data["WARD"] = ward_data["WARD"].astype("string").str.strip()
ward_data = ward_data.dropna(subset=["WARD"])
ward_data = ward_data.loc[
    ~ward_data["WARD"].str.lower().isin(["", "nan", "none", "null"])
].copy()
ward_summary = (
    ward_data.groupby("WARD").agg(
        Total_Parcels=("Log_Appreciation", "size"),
        Mean_Log_Appreciation=("Log_Appreciation", "mean"),
        Median_Log_Appreciation=("Log_Appreciation", "median")
    ).sort_values("Mean_Log_Appreciation", ascending=False)
)
print("Number of WARDs:", len(ward_summary))
print("Overall Mean:", round(ward_data["Log_Appreciation"].mean(), 4))
print("Highest 5 WARDs:")
display(ward_summary.head(5).round(4))
print("Lowest 5 WARDs:")
display(ward_summary.tail(5).round(4))

Number of WARDs: 28
Overall Mean: 0.0784
Highest 5 WARDs:


,Total_Parcels,Mean_Log_Appreciation,Median_Log_Appreciation
WARD,,,
17.0,1432,0.1388,0.0749
10.0,3970,0.1349,0.1048
9.0,3810,0.1291,0.0922
19.0,904,0.1290,0.0723
8.0,3259,0.1256,0.0955


Lowest 5 WARDs:


,Total_Parcels,Mean_Log_Appreciation,Median_Log_Appreciation
WARD,,,
4.0,3496,0.0067,0.0180
21.0,3631,0.0050,0.0170
1.0,3383,-0.0020,0.0173
22.0,2898,-0.0021,0.0183
5.0,602,-0.0166,0.0194


In [38]:
# Step 3.8: Between-WARD Variance Analysis
overall_mean = ward_data["Log_Appreciation"].mean()
total_ss = ((ward_data["Log_Appreciation"] - overall_mean) ** 2).sum()
ward_stats = ward_data.groupby("WARD")["Log_Appreciation"].agg(["mean", "size"])
between_ss = (
    ward_stats["size"] * (ward_stats["mean"] - overall_mean) ** 2
).sum()
within_ss = total_ss - between_ss
ward_eta_squared = between_ss / total_ss
print("Total Sum of Squares:", round(total_ss, 4))
print("Between-WARD Sum of Squares:", round(between_ss, 4))
print("Within-WARD Sum of Squares:", round(within_ss, 4))
print("Variance Explained by WARD:", round(ward_eta_squared, 4))
print("Percentage Explained:", round(ward_eta_squared * 100, 2), "%")

Total Sum of Squares: 3935.8336
Between-WARD Sum of Squares: 197.6501
Within-WARD Sum of Squares: 3738.1835
Variance Explained by WARD: 0.0502
Percentage Explained: 5.02 %


In [41]:
# Compare Raw and Log Appreciation (Primary Residential Sample)
model_data["Log_Appreciation"] = (
    np.log(model_data["Next_ASMTTOTAL"])
    - np.log(model_data["ASMTTOTAL"])
)
display(model_data[["Appreciation", "Log_Appreciation"]].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
))
display(model_data.groupby("Year")["Log_Appreciation"]
        .agg(["count", "mean", "median", "std"]).round(4))

,Appreciation,Log_Appreciation
count,730611.000000,730611.000000
mean,0.106216,0.052509
std,1.368069,0.208561
min,-0.991234,-4.736900
1%,-0.122778,-0.130995
5%,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.076972,0.074154
95%,0.211827,0.192129


,count,mean,median,std
Year,,,,
2017,92669,0.0082,0.0000,0.1784
2018,92235,0.0989,0.0762,0.2023
2019,91724,0.0013,0.0000,0.1425
2020,91605,0.0487,0.0509,0.2122
2021,90747,0.0041,0.0000,0.1396
2022,90805,0.0784,0.0796,0.2082
2023,90465,0.0066,0.0000,0.1289
2024,90361,0.1749,0.1161,0.3229


In [52]:
clean_data = model_data.copy()

print("Original records:", len(model_data))
print("Cleaning records:", len(clean_data))

Original records: 730611
Cleaning records: 730611


In [53]:
clean_data["BDG1YEAR_CLEAN"] = clean_data["BDG1YEAR"].copy()

clean_data.loc[
    (clean_data["BDG1YEAR_CLEAN"] == 0) |
    (clean_data["BDG1YEAR_CLEAN"] > clean_data["Year"]),
    "BDG1YEAR_CLEAN"
] = np.nan

clean_data["Building_Age_CLEAN"] = (
    clean_data["Year"] - clean_data["BDG1YEAR_CLEAN"]
)

display(
    clean_data[["BDG1YEAR_CLEAN", "Building_Age_CLEAN"]].describe()
)

print(
    "Missing Building Year:",
    clean_data["BDG1YEAR_CLEAN"].isna().sum()
)

print(
    "Missing Building Age:",
    clean_data["Building_Age_CLEAN"].isna().sum()
)

print(
    "Negative Building Age:",
    (clean_data["Building_Age_CLEAN"] < 0).sum()
)

,BDG1YEAR_CLEAN,Building_Age_CLEAN
count,720863.000000,720863.000000
mean,1923.066523,97.425834
std,25.369934,25.406734
min,1810.000000,0.000000
25%,1906.000000,84.000000
50%,1922.000000,99.000000
75%,1937.000000,115.000000
max,2024.000000,214.000000


Missing Building Year: 9748
Missing Building Age: 9748
Negative Building Age: 0


In [54]:
clean_data["BDG1AREA_CLEAN"] = clean_data["BDG1AREA"].copy()

mask = (
    (clean_data["VACANTLAND"] == "N") &
    (clean_data["BDG1AREA"] == 0)
)

clean_data.loc[mask, "BDG1AREA_CLEAN"] = np.nan

print("Converted to missing:", mask.sum())
print("Total missing:", clean_data["BDG1AREA_CLEAN"].isna().sum())

Converted to missing: 11726
Total missing: 11726


In [55]:
clean_data["FRONTAGE_CLEAN"] = clean_data["FRONTAGE"].copy()

clean_data["FRONTAGE_ZERO"] = (
    clean_data["FRONTAGE"] == 0
).astype(int)

display(
    clean_data[["BDG1AREA_CLEAN", "FRONTAGE_CLEAN"]]
    .describe()
)

print(
    "Missing BDG1AREA:",
    clean_data["BDG1AREA_CLEAN"].isna().sum()
)

print(
    "Zero FRONTAGE:",
    (clean_data["FRONTAGE_CLEAN"] == 0).sum()
)

print(
    "Total records:",
    len(clean_data)
)

,BDG1AREA_CLEAN,FRONTAGE_CLEAN
count,718885.000000,730611.000000
mean,1687.950953,35.242220
std,1436.882953,15.335541
min,100.000000,0.000000
25%,960.000000,28.000000
50%,1308.000000,33.000000
75%,2136.000000,40.010000
max,189868.000000,350.940000


Missing BDG1AREA: 11726
Zero FRONTAGE: 31584
Total records: 730611


In [56]:
clean_data["NUMBLDGS_CLEAN"] = clean_data["NUMBLDGS"].copy()

mask = (
    (clean_data["VACANTLAND"] == "N") &
    (clean_data["NUMBLDGS"] == 0)
)

clean_data.loc[mask, "NUMBLDGS_CLEAN"] = np.nan

print("Converted to missing:", mask.sum())

clean_data["NUMUNITS_CLEAN"] = clean_data["NUMUNITS"].copy()

mask = (
    (clean_data["VACANTLAND"] == "Y") &
    (clean_data["NUMBLDGS"] == 0) &
    (clean_data["BDG1AREA"] == 0) &
    (clean_data["NUMUNITS"] >= 1000)
)

clean_data.loc[mask, "NUMUNITS_CLEAN"] = np.nan

print("Converted to missing:", mask.sum())

display(
    clean_data[["NUMBLDGS_CLEAN", "NUMUNITS_CLEAN"]]
    .describe()
)

print(
    "Missing NUMBLDGS:",
    clean_data["NUMBLDGS_CLEAN"].isna().sum()
)

print(
    "Missing NUMUNITS:",
    clean_data["NUMUNITS_CLEAN"].isna().sum()
)

print(
    "Total records:",
    len(clean_data)
)

Converted to missing: 9656
Converted to missing: 0


,NUMBLDGS_CLEAN,NUMUNITS_CLEAN
count,720955.000000,730611.000000
mean,1.013287,1.753543
std,0.279799,8.032748
min,1.000000,0.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,1.000000,1.000000
max,59.000000,2755.000000


Missing NUMBLDGS: 9656
Missing NUMUNITS: 0
Total records: 730611


In [57]:
# Clean Categorical Variables — preserve true original missingness
cat_variables = ["LANDUSE1", "ASRCLASS1", "ZONING1"]

for col in cat_variables:
    original = clean_data[col].astype("string").str.strip()
    # Treat empty and placeholder strings as missing, not as legitimate categories.
    missing = original.isna() | original.str.lower().isin(
        ["", "nan", "none", "null", "<na>"]
    ).fillna(False)
    clean_data[f"{col}_MISSING"] = missing.astype(int)
    cleaned = original.mask(missing, "Unknown")
    # Normalize numeric category codes like '1110.0' and '1110'.
    if col in ["LANDUSE1", "ASRCLASS1"]:
        cleaned = cleaned.str.replace(r"^(\d+)\.0$", r"\1", regex=True)
    clean_data[f"{col}_CLEAN"] = cleaned.fillna("Unknown").astype(str)

clean_data["VACANTLAND_CLEAN"] = (
    clean_data["VACANTLAND"].astype("string").str.strip().str.upper()
    .map({"N": 0, "Y": 1})
)

cat_cols = [
    "LANDUSE1_CLEAN", "ASRCLASS1_CLEAN",
    "ZONING1_CLEAN", "VACANTLAND_CLEAN"
]
display(clean_data[cat_cols].head())
print("\nMissing values AFTER categorical encoding:")
display(clean_data[cat_cols].isna().sum())
print("\nMissing or placeholder values BEFORE encoding:")
display(clean_data[[f"{col}_MISSING" for col in cat_variables]].sum())
print("\nUnique categories:")
display(clean_data[cat_cols].nunique())

,LANDUSE1_CLEAN,ASRCLASS1_CLEAN,ZONING1_CLEAN,VACANTLAND_CLEAN
0,1110,200,L,0
1,1185,200,I,0
2,1110,200,I,0
3,1115,200,I,0
4,1185,100,I,0



Missing values AFTER categorical encoding:


LANDUSE1_CLEAN      0
ASRCLASS1_CLEAN     0
ZONING1_CLEAN       0
VACANTLAND_CLEAN    0
dtype: int64


Missing or placeholder values BEFORE encoding:


LANDUSE1_MISSING      0
ASRCLASS1_MISSING     0
ZONING1_MISSING      19
dtype: int64


Unique categories:


LANDUSE1_CLEAN      27
ASRCLASS1_CLEAN     44
ZONING1_CLEAN       13
VACANTLAND_CLEAN     1
dtype: int64

In [58]:
# Feature Lists and Missingness Audit
numeric_features = [
    "ASMTTOTAL", "Building_Age_CLEAN", "BDG1AREA_CLEAN",
    "FRONTAGE_CLEAN", "FRONTAGE_ZERO",
    "NUMBLDGS_CLEAN", "NUMUNITS_CLEAN"
]

# VACANTLAND_CLEAN is always 0 in the primary eligible sample (VACANTLAND == N);
# preserve it for audits, but exclude it from the model features.
categorical_features = [
    "LANDUSE1_CLEAN", "ASRCLASS1_CLEAN", "ZONING1_CLEAN"
]
features = numeric_features + categorical_features

missing_summary = pd.DataFrame({
    "Missing": clean_data[features].isna().sum(),
    "Missing_Percent": clean_data[features].isna().mean() * 100
})
display(missing_summary.round(2))
display(clean_data[["Appreciation", "Log_Appreciation"]].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
))
print("Missing Appreciation:", clean_data["Appreciation"].isna().sum())
print("Missing Log Appreciation:", clean_data["Log_Appreciation"].isna().sum())
print("Infinite Log Appreciation:", np.isinf(clean_data["Log_Appreciation"]).sum())

,Missing,Missing_Percent
ASMTTOTAL,0,0.00
Building_Age_CLEAN,9748,1.33
BDG1AREA_CLEAN,11726,1.60
FRONTAGE_CLEAN,0,0.00
FRONTAGE_ZERO,0,0.00
NUMBLDGS_CLEAN,9656,1.32
NUMUNITS_CLEAN,0,0.00
LANDUSE1_CLEAN,0,0.00
ASRCLASS1_CLEAN,0,0.00
ZONING1_CLEAN,0,0.00


,Appreciation,Log_Appreciation
count,730611.000000,730611.000000
mean,0.106216,0.052509
std,1.368069,0.208561
min,-0.991234,-4.736900
1%,-0.122778,-0.130995
5%,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.076972,0.074154
95%,0.211827,0.192129


Missing Appreciation: 0
Missing Log Appreciation: 0
Infinite Log Appreciation: 0


In [62]:
# Chronological Train / Validation / Final-Test Split
# Training: 2017–2022 targets; validation: 2023→2024;
# final test: 2024→2025
train_data = clean_data.loc[clean_data["Year"].between(2017, 2022)].copy()
val_data = clean_data.loc[clean_data["Year"] == 2023].copy()
test_data = clean_data.loc[clean_data["Year"] == 2024].copy()

print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Testing (held out):", len(test_data))

assert train_data["Year"].max() < val_data["Year"].min()
assert val_data["Year"].max() < test_data["Year"].min()

# Summarize training/validation only. No final-test targets here.
display(pd.DataFrame({
    "Train": train_data["Log_Appreciation"].describe(),
    "Validation": val_data["Log_Appreciation"].describe()
}).round(4))

Training: 549785
Validation: 90465
Testing (held out): 90361


,Train,Validation
count,549785.0000,90465.0000
mean,0.0399,0.0066
std,0.1870,0.1289
min,-4.7369,-2.9054
25%,0.0000,0.0000
50%,0.0000,0.0000
75%,0.0662,0.0000
max,5.2764,4.5167


In [63]:
# Train-only Feature Preprocessing (No Future Leakage)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train = train_data[features]
X_val = val_data[features]
X_test = test_data[features]  # Kept for a later final test; not evaluated now.

y_train = train_data["Log_Appreciation"]
y_val = val_data["Log_Appreciation"]
# Intentionally do NOT create or evaluate y_test during development.

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Number of features:", len(preprocessor.get_feature_names_out()))

Training: (549785, 91)
Validation: (90465, 91)
Number of features: 91


In [65]:
# Step 4.1: Zero-Change Baseline (predict log appreciation = 0)
# Use the same held-out 2023 validation rows as the other two baselines.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
zero_pred = np.zeros(len(y_val))
print("Zero-Change Baseline (2023 Validation)")
print("MAE:", round(mean_absolute_error(y_val, zero_pred), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, zero_pred)), 4))
print("R2:", round(r2_score(y_val, zero_pred), 4))

Zero-Change Baseline (2023 Validation)
MAE: 0.0091
RMSE: 0.1291
R2: -0.0026


In [66]:
# Mean baseline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mean_pred = np.full(len(y_val), y_train.mean())

In [67]:
# Linear Regression
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train_processed, y_train)

lr_pred = lr.predict(X_val_processed)

In [68]:
# Compare all three baselines on the SAME 2023 residential validation set
results = pd.DataFrame({
    "Model": ["Zero-Change Baseline", "Mean Baseline", "Linear Regression"],
    "MAE": [
        mean_absolute_error(y_val, zero_pred),
        mean_absolute_error(y_val, mean_pred),
        mean_absolute_error(y_val, lr_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, zero_pred)),
        np.sqrt(mean_squared_error(y_val, mean_pred)),
        np.sqrt(mean_squared_error(y_val, lr_pred))
    ],
    "R2": [
        r2_score(y_val, zero_pred),
        r2_score(y_val, mean_pred),
        r2_score(y_val, lr_pred)
    ]
}).sort_values("MAE", ascending=True).reset_index(drop=True)
display(results.round(4))

,Model,MAE,RMSE,R2
0,Zero-Change Baseline,0.0091,0.1291,-0.0026
1,Linear Regression,0.0478,0.1330,-0.0643
2,Mean Baseline,0.0482,0.1332,-0.0669


In [133]:
# Last-Observed-Change Baseline
# Find the latest historical change for each property
last_history = (
    train_data
    .sort_values(["HANDLE", "Year"])
    .drop_duplicates("HANDLE", keep="last")
    .set_index("HANDLE")
)

# Match historical changes to 2023 validation properties
last_pred_series = val_data["HANDLE"].map(
    last_history["Log_Appreciation"]
)

# Use zero if historical change is unavailable
last_pred = last_pred_series.fillna(0).to_numpy()

# Evaluate the baseline
last_result = pd.DataFrame({
    "Model": ["Last-Observed-Change Baseline"],
    "MAE": [mean_absolute_error(y_val, last_pred)],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, last_pred))
    ],
    "R2": [r2_score(y_val, last_pred)]
})

# Compare four models
results_4_2 = pd.concat(
    [results, last_result],
    ignore_index=True
).sort_values("MAE").reset_index(drop=True)

print("Properties with historical changes:",
      last_pred_series.notna().sum())

print("Properties using zero fallback:",
      last_pred_series.isna().sum())

display(results_4_2.round(4))

Properties with historical changes: 90324
Properties using zero fallback: 141


,Model,MAE,RMSE,R2
0,Zero-Change Baseline,0.0091,0.1291,-0.0026
1,Linear Regression,0.0478,0.1330,-0.0643
2,Mean Baseline,0.0482,0.1332,-0.0669
3,Last-Observed-Change Baseline,0.0957,0.2371,-2.3820


In [137]:
# Cycle-Aware Historical Baseline
# Historical log appreciation lookup
history_lookup = train_data.set_index(
    ["HANDLE", "Year"]
)["Log_Appreciation"]

# Match each validation property to two years earlier
history_keys = pd.MultiIndex.from_arrays(
    [
        val_data["HANDLE"],
        val_data["Year"] - 2
    ],
    names=["HANDLE", "Year"]
)

cycle_pred_series = history_lookup.reindex(history_keys)

# Use zero if same-cycle history is unavailable
cycle_pred = cycle_pred_series.fillna(0).to_numpy()

# Evaluate
cycle_result = pd.DataFrame({
    "Model": ["Cycle-Aware Historical Baseline"],
    "MAE": [mean_absolute_error(y_val, cycle_pred)],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val, cycle_pred))
    ],
    "R2": [r2_score(y_val, cycle_pred)]
})

# Compare all five models
results_4_3 = pd.concat(
    [results_4_2, cycle_result],
    ignore_index=True
).sort_values("MAE").reset_index(drop=True)

print("Properties with same-cycle history:",
      cycle_pred_series.notna().sum())

print("Properties using zero fallback:",
      cycle_pred_series.isna().sum())

display(results_4_3.round(4))

Properties with same-cycle history: 89777
Properties using zero fallback: 688


,Model,MAE,RMSE,R2
0,Zero-Change Baseline,0.0091,0.1291,-0.0026
1,Cycle-Aware Historical Baseline,0.0174,0.1792,-0.9314
2,Linear Regression,0.0478,0.1330,-0.0643
3,Mean Baseline,0.0482,0.1332,-0.0669
4,Last-Observed-Change Baseline,0.0957,0.2371,-2.3820


In [139]:
# Set Up Rolling-Origin Validation
rolling_splits = []

for val_year in range(2020, 2024):

    # Training uses only earlier years
    rolling_train = clean_data[
        clean_data["Year"] < val_year
    ].copy()

    # Validate on one future year
    rolling_val = clean_data[
        clean_data["Year"] == val_year
    ].copy()

    rolling_splits.append({
        "Validation_Year": val_year,
        "Training_Years": f"2017-{val_year - 1}",
        "Target_Year": val_year + 1,
        "Assessment_Cycle": (
            "Reassessment"
            if (val_year + 1) % 2 == 1
            else "Non-Reassessment"
        ),
        "Train_Size": len(rolling_train),
        "Validation_Size": len(rolling_val)
    })

rolling_summary = pd.DataFrame(rolling_splits)

display(rolling_summary)

,Validation_Year,Training_Years,Target_Year,Assessment_Cycle,Train_Size,Validation_Size
0,2020,2017-2019,2021,Reassessment,276628,91605
1,2021,2017-2020,2022,Non-Reassessment,368233,90747
2,2022,2017-2021,2023,Reassessment,458980,90805
3,2023,2017-2022,2024,Non-Reassessment,549785,90465


In [141]:
# Rolling-Origin Baseline Evaluation
rolling_results = []

for val_year in range(2020, 2024):

    # Chronological training and validation
    fold_train = clean_data[
        clean_data["Year"] < val_year
    ].copy()

    fold_val = clean_data[
        clean_data["Year"] == val_year
    ].copy()

    y_train_fold = fold_train["Log_Appreciation"]
    y_val_fold = fold_val["Log_Appreciation"]

    # 1. Zero-Change Baseline
    zero_pred = np.zeros(len(fold_val))

    # 2. Training Mean Baseline
    mean_pred = np.full(
        len(fold_val),
        y_train_fold.mean()
    )

    # 3. Last-Observed-Change Baseline
    last_history = (
        fold_train
        .sort_values(["HANDLE", "Year"])
        .drop_duplicates("HANDLE", keep="last")
        .set_index("HANDLE")["Log_Appreciation"]
    )

    last_pred = (
        fold_val["HANDLE"]
        .map(last_history)
        .fillna(0)
        .to_numpy()
    )

    # 4. Cycle-Aware Historical Baseline
    cycle_history = (
        fold_train[
            fold_train["Year"] == val_year - 2
        ]
        .set_index("HANDLE")["Log_Appreciation"]
    )

    cycle_pred = (
        fold_val["HANDLE"]
        .map(cycle_history)
        .fillna(0)
        .to_numpy()
    )

    predictions = {
        "Zero-Change": zero_pred,
        "Mean Baseline": mean_pred,
        "Last-Observed-Change": last_pred,
        "Cycle-Aware Historical": cycle_pred
    }

    # Evaluate every baseline
    for model_name, pred in predictions.items():

        rolling_results.append({
            "Validation_Year": val_year,
            "Assessment_Cycle": (
                "Reassessment"
                if (val_year + 1) % 2 == 1
                else "Non-Reassessment"
            ),
            "Model": model_name,
            "MAE": mean_absolute_error(
                y_val_fold, pred
            ),
            "RMSE": np.sqrt(
                mean_squared_error(y_val_fold, pred)
            ),
            "R2": r2_score(y_val_fold, pred)
        })

rolling_results = pd.DataFrame(rolling_results)

display(
    rolling_results.sort_values(
        ["Validation_Year", "MAE"]
    ).round(4)
)

,Validation_Year,Assessment_Cycle,Model,MAE,RMSE,R2
1,2020,Reassessment,Mean Baseline,0.0638,0.2126,-0.0035
0,2020,Reassessment,Zero-Change,0.0753,0.2178,-0.0526
2,2020,Reassessment,Last-Observed-Change,0.0860,0.2689,-0.6051
3,2020,Reassessment,Cycle-Aware Historical,0.1168,0.2867,-0.8242
4,2021,Non-Reassessment,Zero-Change,0.0105,0.1397,-0.0009
7,2021,Non-Reassessment,Cycle-Aware Historical,0.0191,0.1856,-0.7657
5,2021,Non-Reassessment,Mean Baseline,0.0490,0.1440,-0.0636
6,2021,Non-Reassessment,Last-Observed-Change,0.0771,0.2306,-1.7273
9,2022,Reassessment,Mean Baseline,0.0696,0.2132,-0.0490
11,2022,Reassessment,Cycle-Aware Historical,0.0725,0.2673,-0.6489


In [143]:
# Rolling-Origin Linear Regression
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

rolling_lr_results = []

for val_year in range(2020, 2024):

    # Training and validation data
    fold_train = clean_data[
        clean_data["Year"] < val_year
    ].copy()

    fold_val = clean_data[
        clean_data["Year"] == val_year
    ].copy()

    X_train_fold = fold_train[features]
    y_train_fold = fold_train["Log_Appreciation"]

    X_val_fold = fold_val[features]
    y_val_fold = fold_val["Log_Appreciation"]

    # Fresh preprocessing and regression for each fold
    fold_model = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("regression", LinearRegression())
    ])

    # Fit using earlier years only
    fold_model.fit(X_train_fold, y_train_fold)

    # Predict the validation year
    lr_pred_fold = fold_model.predict(X_val_fold)

    rolling_lr_results.append({
        "Validation_Year": val_year,
        "Assessment_Cycle": (
            "Reassessment"
            if (val_year + 1) % 2 == 1
            else "Non-Reassessment"
        ),
        "Model": "Linear Regression",
        "MAE": mean_absolute_error(
            y_val_fold, lr_pred_fold
        ),
        "RMSE": np.sqrt(
            mean_squared_error(y_val_fold, lr_pred_fold)
        ),
        "R2": r2_score(
            y_val_fold, lr_pred_fold
        )
    })

rolling_lr_results = pd.DataFrame(
    rolling_lr_results
)

# Combine with the four previous baselines
rolling_results_all = pd.concat(
    [rolling_results, rolling_lr_results],
    ignore_index=True
)

display(
    rolling_results_all.sort_values(
        ["Validation_Year", "MAE"]
    ).round(4)
)

,Validation_Year,Assessment_Cycle,Model,MAE,RMSE,R2
1,2020,Reassessment,Mean Baseline,0.0638,0.2126,-0.0035
16,2020,Reassessment,Linear Regression,0.0638,0.2128,-0.0053
0,2020,Reassessment,Zero-Change,0.0753,0.2178,-0.0526
2,2020,Reassessment,Last-Observed-Change,0.0860,0.2689,-0.6051
3,2020,Reassessment,Cycle-Aware Historical,0.1168,0.2867,-0.8242
4,2021,Non-Reassessment,Zero-Change,0.0105,0.1397,-0.0009
7,2021,Non-Reassessment,Cycle-Aware Historical,0.0191,0.1856,-0.7657
17,2021,Non-Reassessment,Linear Regression,0.0489,0.1440,-0.0632
5,2021,Non-Reassessment,Mean Baseline,0.0490,0.1440,-0.0636
6,2021,Non-Reassessment,Last-Observed-Change,0.0771,0.2306,-1.7273


In [145]:
# Top-K Property Ranking Evaluation

screening_budget = 0.05
top_outcome_fraction = 0.10

ranking_results = []

for val_year in [2020, 2022]:

    # Training and validation
    fold_train = clean_data[
        clean_data["Year"] < val_year
    ].copy()

    fold_val = clean_data[
        clean_data["Year"] == val_year
    ].copy()

    X_train_fold = fold_train[features]
    y_train_fold = fold_train["Log_Appreciation"]

    X_val_fold = fold_val[features]
    y_val_fold = fold_val["Log_Appreciation"]

    # Train Linear Regression
    model = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("regression", LinearRegression())
    ])

    model.fit(X_train_fold, y_train_fold)

    predictions = model.predict(X_val_fold)
    actual = y_val_fold.to_numpy()

    n = len(fold_val)

    # Number of properties that can be investigated
    k = max(1, int(np.ceil(n * screening_budget)))

    # Actual top 10% appreciation
    top_n = max(
        1, int(np.ceil(n * top_outcome_fraction))
    )

    # Select top 5% by model predictions
    selected = np.zeros(n, dtype=bool)

    selected[
        np.argsort(-predictions, kind="stable")[:k]
    ] = True

    # Identify actual top 10%
    strong = np.zeros(n, dtype=bool)

    strong[
        np.argsort(-actual, kind="stable")[:top_n]
    ] = True

    # Precision at screening budget
    hits = (selected & strong).sum()

    precision_at_k = hits / k

    # Expected precision under random screening
    random_precision = strong.mean()

    # Improvement over random screening
    lift = precision_at_k / random_precision

    ranking_results.append({
        "Validation_Year": val_year,
        "Candidates": n,
        "Screening_Budget": k,
        "Top_10%_Found": hits,
        "Precision_at_5%": precision_at_k,
        "Random_Precision": random_precision,
        "Lift_vs_Random": lift,
        "Mean_Selected_Log": actual[selected].mean(),
        "Mean_Overall_Log": actual.mean()
    })

ranking_results = pd.DataFrame(ranking_results)

display(ranking_results.round(4))

,Validation_Year,Candidates,Screening_Budget,Top_10%_Found,Precision_at_5%,Random_Precision,Lift_vs_Random,Mean_Selected_Log,Mean_Overall_Log
0,2020,91605,4581,191,0.0417,0.1,0.4169,0.0188,0.0487
1,2022,90805,4541,236,0.0520,0.1,0.5197,0.1174,0.0784


In [147]:
# Historical Ranking Baseline
historical_ranking_results = []

for val_year in [2020, 2022]:

    fold_train = clean_data[
        clean_data["Year"] < val_year
    ].copy()

    fold_val = clean_data[
        clean_data["Year"] == val_year
    ].copy()

    # Use historical changes from two years earlier
    history = (
        fold_train[
            fold_train["Year"] == val_year - 2
        ]
        .set_index("HANDLE")["Log_Appreciation"]
    )

    historical_scores = (
        fold_val["HANDLE"]
        .map(history)
        .fillna(0)
        .to_numpy()
    )

    actual = fold_val["Log_Appreciation"].to_numpy()
    n = len(actual)

    k = int(np.ceil(n * 0.05))
    top_n = int(np.ceil(n * 0.10))

    # Top 5% by historical appreciation
    selected_idx = np.argsort(
        -historical_scores, kind="stable"
    )[:k]

    # Actual top 10% appreciation
    actual_top_idx = np.argsort(
        -actual, kind="stable"
    )[:top_n]

    hits = np.isin(
        selected_idx, actual_top_idx
    ).sum()

    precision = hits / k
    random_precision = top_n / n

    historical_ranking_results.append({
        "Validation_Year": val_year,
        "Model": "Historical Ranking",
        "Precision_at_5%": precision,
        "Lift_vs_Random": precision / random_precision,
        "Mean_Selected_Log": actual[selected_idx].mean()
    })

historical_ranking_results = pd.DataFrame(
    historical_ranking_results
)

# Compare historical ranking with Linear Regression
lr_ranking = ranking_results[
    [
        "Validation_Year",
        "Precision_at_5%",
        "Lift_vs_Random",
        "Mean_Selected_Log"
    ]
].copy()

lr_ranking["Model"] = "Linear Regression"

ranking_comparison = pd.concat(
    [lr_ranking, historical_ranking_results],
    ignore_index=True
)

display(
    ranking_comparison.sort_values(
        ["Validation_Year", "Model"]
    ).round(4)
)

,Validation_Year,Precision_at_5%,Lift_vs_Random,Mean_Selected_Log,Model
2,2020,0.1637,1.6371,0.0633,Historical Ranking
0,2020,0.0417,0.4169,0.0188,Linear Regression
3,2022,0.0707,0.7069,0.0991,Historical Ranking
1,2022,0.0520,0.5197,0.1174,Linear Regression


In [173]:
# Class 100 Residential Screening
class100_results = []

# Normalize assessment-class codes
class_codes = (
    clean_data["ASRCLASS1_CLEAN"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

for val_year in [2020, 2022]:

    # Training: only earlier years
    fold_train = clean_data[
        clean_data["Year"] < val_year
    ].copy()

    # Validation: Class 100 residential properties
    fold_val = clean_data[
        (clean_data["Year"] == val_year) &
        (class_codes == "100")
    ].copy()

    # Historical change from the same assessment cycle
    history = (
        fold_train[
            fold_train["Year"] == val_year - 2
        ]
        .drop_duplicates("HANDLE")
        .set_index("HANDLE")["Log_Appreciation"]
    )

    history_raw = fold_val["HANDLE"].map(history)

    scores = history_raw.fillna(0).to_numpy()
    actual = fold_val["Log_Appreciation"].to_numpy()

    n = len(fold_val)

    # Actual top 10% within Class 100
    top_n = int(np.ceil(n * 0.10))

    strong = np.zeros(n, dtype=bool)

    strong[
        np.argsort(-actual, kind="stable")[:top_n]
    ] = True

    # Rank using historical appreciation
    ranked_idx = np.argsort(
        -scores, kind="stable"
    )

    for budget in [0.01, 0.05]:

        k = int(np.ceil(n * budget))
        selected_idx = ranked_idx[:k]

        hits = int(strong[selected_idx].sum())

        precision = hits / k
        random_rate = top_n / n

        # Check ties at the selection boundary
        cutoff = scores[ranked_idx[k - 1]]

        ties_cross_boundary = (
            (scores > cutoff).sum() < k
            and (scores >= cutoff).sum() > k
        )

        class100_results.append({
            "Validation_Year": val_year,
            "Model": "Class 100 Historical Ranking",
            "Candidates": n,
            "Budget": budget,
            "Investigated": k,
            "Hits": hits,
            "Precision": precision,
            "Lift_vs_Random": precision / random_rate,
            "Missing_History_Pct": (
                100 * history_raw.isna().mean()
            ),
            "Boundary_Ties": ties_cross_boundary
        })

class100_results = pd.DataFrame(class100_results)

display(class100_results.round(4))

,Validation_Year,Model,Candidates,Budget,Investigated,Hits,Precision,Lift_vs_Random,Missing_History_Pct,Boundary_Ties
0,2020,Class 100 Historical Ranking,87621,0.01,877,145,0.1653,1.6532,0.6768,False
1,2020,Class 100 Historical Ranking,87621,0.05,4382,728,0.1661,1.6612,0.6768,False
2,2022,Class 100 Historical Ranking,88123,0.01,882,109,0.1236,1.2357,0.7478,False
3,2022,Class 100 Historical Ranking,88123,0.05,4407,307,0.0697,0.6966,0.7478,False


In [179]:
# Class 100 Screening List with Risk Flags
val_year = 2022

# Select Class 100 residential candidates
class_code = (
    clean_data["ASRCLASS1_CLEAN"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

candidates = clean_data.loc[
    (clean_data["Year"] == val_year) &
    (class_code == "100")
].copy().reset_index(drop=True)

# Retrieve same-cycle historical information
history = (
    clean_data[
        clean_data["Year"] == val_year - 2
    ]
    .drop_duplicates("HANDLE")
    .set_index("HANDLE")
)

candidates["Historical_Log_Change"] = (
    candidates["HANDLE"]
    .map(history["Log_Appreciation"])
)

previous_class = (
    candidates["HANDLE"]
    .map(history["ASRCLASS1_CLEAN"])
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

# Create independent assessment risk flags
candidates["Extreme_History"] = (
    candidates["Historical_Log_Change"].abs()
    > np.log(2)
).fillna(False)

candidates["Class_Changed"] = (
    previous_class.notna() &
    (previous_class != "100")
).fillna(False)

candidates["Missing_History"] = (
    candidates["Historical_Log_Change"].isna()
)

# Preserve original historical ranking rule
scores = (
    candidates["Historical_Log_Change"]
    .fillna(0)
    .to_numpy()
)

ranked_idx = np.argsort(
    -scores, kind="stable"
)

screening_class100 = (
    candidates.iloc[ranked_idx]
    .copy()
    .reset_index(drop=True)
)

screening_class100["Rank"] = (
    np.arange(1, len(screening_class100) + 1)
)

# Assign screening priority
n = len(screening_class100)
k1 = int(np.ceil(n * 0.01))
k5 = int(np.ceil(n * 0.05))

screening_class100["Priority"] = np.select(
    [
        screening_class100["Rank"] <= k1,
        screening_class100["Rank"] <= k5
    ],
    ["Consider", "Review"],
    default="Defer"
)

# Add assessment verification status
risk_columns = [
    "Extreme_History",
    "Class_Changed",
    "Missing_History"
]

screening_class100["Needs_Verification"] = (
    screening_class100[risk_columns].any(axis=1)
)

# Display summaries
print("Screening Priority Counts:")

display(
    screening_class100["Priority"]
    .value_counts()
    .reindex(["Consider", "Review", "Defer"])
)

print("Risk Flags in Consider Group:")

consider_class100 = screening_class100[
    screening_class100["Priority"] == "Consider"
]

display(
    consider_class100[
        risk_columns + ["Needs_Verification"]
    ].sum()
)

print("Top 10 Residential Candidates:")

display(
    screening_class100[
        [
            "HANDLE",
            "ASMTTOTAL",
            "Historical_Log_Change",
            "Rank",
            "Priority",
            "Extreme_History",
            "Class_Changed",
            "Needs_Verification"
        ]
    ].head(10)
)

Screening Priority Counts:


Priority
Consider      882
Review       3525
Defer       83716
Name: count, dtype: int64

Risk Flags in Consider Group:


Extreme_History       319
Class_Changed          73
Missing_History         0
Needs_Verification    326
dtype: Int64

Top 10 Residential Candidates:


,HANDLE,ASMTTOTAL,Historical_Log_Change,Rank,Priority,Extreme_History,Class_Changed,Needs_Verification
0,1.397300e+10,69810.0,4.773410,1,Consider,True,False,True
1,1.475304e+10,56660.0,4.467851,2,Consider,True,False,True
2,1.475304e+10,56410.0,4.463429,3,Consider,True,False,True
3,1.475304e+10,56410.0,4.463429,4,Consider,True,False,True
4,1.113600e+10,13810.0,4.095069,5,Consider,True,True,True
5,1.113600e+10,13810.0,4.095069,6,Consider,True,True,True
6,1.113600e+10,13810.0,4.095069,7,Consider,True,True,True
7,1.114000e+10,13430.0,4.067167,8,Consider,True,True,True
8,1.113600e+10,13810.0,4.011687,9,Consider,True,True,True
9,1.457200e+10,62550.0,3.987546,10,Consider,True,True,True
